In [ ]:
#Python Libraries

import numpy as np # linear algebra
import pandas as pd # data processing

import matplotlib.pyplot as plt # data visualization
from pandas.plotting import scatter_matrix # data visualization

from sklearn.model_selection import train_test_split # Machine Learning - split dataset (train/test)
from sklearn.preprocessing import OrdinalEncoder # Machine Learning (clean data) - Encoder Categorical Attributes
from sklearn.preprocessing import OneHotEncoder # Machine Learning (clean data)- OneHot Encoder Categorical Attributes
from sklearn.preprocessing import MinMaxScaler #Machine Learning (scaling data) - Normalization
from sklearn.preprocessing import StandardScaler #Machine Learning (scaling data) - Z-Score Normalization
from sklearn.linear_model import LinearRegression #Machine Learning - Linear Regression Model
from sklearn.metrics import mean_squared_error #Machine Learning - Mean Square Error (MSE)
from sklearn.model_selection import cross_val_score #Machine Learning - Evaluation by Cross Validation

# __0. Important__

Type of machine learning system to build:

1. Supervised Learning
2. Batch Learning (also called "offline learning")
3. Model-based learning


# __1. Loading the Data__

Showing only the first ten rows of the Data (the first row is the head, and the following rows are data points).

In [ ]:
training_data = pd.read_csv("/kaggle/input/titanic/train.csv")
training_data.head(10)

# __2. Exploring the Data__

## __2.1. Looking at the Data Structure__

### __2.1.1. Getting information/description of data__:
1. Number of rows and columns
2. Number and names of features' type (float64, int64, object)
3. Number of non-null values
4. Name of features

In [ ]:
training_data.info()

### __2.1.2. Data Dictionary__:

1. survival - Survival (0 = No, 1 = Yes)
2. pclass - Ticket class (1 = 1st, 2 = 2nd, 3 = 3rd)
3. sex - Sex 
4. Age - Age in years
5. sibsp - number of siblings / spouses aboard the Titanic.
        5.1. Sibling = brother, sister, stepbrother, stepsister.
        5.2. Spouse = husband, wife (mistresses and fiancés were ignored).
6. parch - number of parents / children aboard the Titanic. The dataset defines family relations in this way:
        6.1. Parent = mother, father
        6.2. Child = daughter, son, stepdaughter, stepson
        6.3. Some children travelled only with a nanny, therefore parch=0 for them.
7. ticket - Ticket number
8. fare - Passenger fare
9. cabin - Cabin number
10. embarked - Port of Embarkation (C = Cherbourg, Q = Queenstown, S = Southampton)

### __2.1.3. Categories that exist in a feature__:

After checking the first rows of data, we know the following about the features "Sex" and "Embarked":
1. They are categorical attributes (object)
2. They have repetitive values
3. They have several categories

After checking the first rows of data, we know the following about the features "Pclass", "SibSp", and "Parch":
1. They are categorical attributes (int64)
2. They have repetitive values
3. They have several categories

In [ ]:
#Categories in a feature.
training_data["Sex"].value_counts()

In [ ]:
#Categories in a feature.
training_data["Embarked"].value_counts()

In [ ]:
#Categories in a feature.
training_data["Pclass"].value_counts()

In [ ]:
#Categories in a feature.
training_data["SibSp"].value_counts()

In [ ]:
#Categories in a feature.
training_data["Parch"].value_counts()

### __2.1.4. Summary of numerical attributes__:

In essence, it is descriptive statistics (count, mean, standard deviation, minimum value, maximum value, and 25% - 50% - 75% percentile).

<div class="alert alert-block alert-warning"><b>Important:</b> After providing a brief overview of values such as the "mean, min, and max", we can conclude that it will be necessary to apply "feature scaling" soon in some numeric attributes. 

For instance, we can see that the mean values of features such as "Fare" and "Age" are much larger than the mean value of the feature "Pclass" .</div>

In [ ]:
training_data.describe()

### __2.1.5. Summary of numerical attributes (in a graph way)__:

Histograms and BoxPlots applied for each numerical attribute.

The data have not been scaled or capped yet. 

> There are no preprocessed attributes.

<div class="alert alert-block alert-warning"><b>Important:</b> Be cautious of histograms that are tail-heavy, such as those belonging to the features "Fare" and "Age". Consider using feature scaling techniques to transform them into more bell-shaped distributions. </div>

In [ ]:
#Histogram

    #The argument "Figsize" is used for adjusting the size values of a graph (x,y).

training_data.hist(bins= 50, figsize = (8,8))
plt.show

In [ ]:
#BoxPlot

    #The argument "Figsize" is used for adjusting the size values of a graph (x,y).

boxplot = training_data.boxplot(column=["PassengerId",'Survived','Pclass',"Age", "SibSp", "Parch", "Fare"],figsize = (8,8)) 
plt.show

# __3. Creating a Test Set__

Although this competition has a separate Test Set, I have decided to set aside from the "Training Set" another "Test Set". It is theoretically simple picking 20% of the dataset.

There are a couple of ways to create a Test Set. They are the followings:
1. __Option 1__: shuffling indices and then splitting data randomly.

> <span style="color:black">__Caution:__</span> This option (purely random sampling) is appropriate if the dataset is large enough relative to the number of attributes.

> No established "random number generator seed" to generate the same shuffled indices for the Test Set could be a problem. With this function, the generation of different Test Sets every time we run is avoided.

> It could be problematic if the dataset is updated. Maintaining stable the "training and test data" after the dataset is updated is achievable if we establish identifiers for the data points that will be part of the training and test set. <-- <span style="color:black">__It is NOT DONE in this Notebook.__</span> 
        

2. __Option 2__: Using the Scikit Learn function to split data randomly.

> <span style="color:black">__Caution:__</span> This option (purely random sampling) is appropriate if the dataset is large enough relative to the number of attributes.

> This function allows splitting the data into training data and a test set. In addition, it has the parameter to set the "random number generator seed" called "random_state"

3. __Option 3__: Using the "Scikit Learn" function to split the data in a stratified way. <-- <span style="color:black">__This option is NOT DONE in this Notebook.__</span> 

> <span style="color:black">__Caution:__</span> This option (stratified sampling) is appropriate if the dataset is not large enough relative to the number of attributes; therefore, it is avoided the risk of sampling bias.

> The purpose of this is to split the entire dataset into homogeneous groups. Then from those groups, we sample the right amount of items that represent each of them fairly. Finally, those samples are going to be part of the Test Set (i.e., the Test Set is going to have representative data from each group of the entire dataset).

> <span style="color:black">__Caution:__</span> Do not forget to deal with or preprocess NA values before applying this procedure.

## __3.1. Option 1: Shuffling indices and then splitting data randomly__

In [ ]:
#"random number generator seed"
np.random.seed(66)

#Function to split the dataset 
    #The role of indices in this function is relevant. 
    #np.random.permutation(x): Randomly permute a sequence, or return a permuted range. 
        #If x is a multi-dimensional array, it is only shuffled along its first index.
        #If x is an array, make a copy and shuffle the elements randomly.
    #iloc: integer-location based indexing for selection by position.
        #Indexing just the rows that belong to the "training set" and "test set"
    
def split_dataset(dataset, test_ratio):
    shuffled_indices_dataset = np.random.permutation(len(dataset))
    size_of_test_set = int(len(dataset)*test_ratio)
    indices_of_test_set = shuffled_indices_dataset[:size_of_test_set]
    indices_of_train_set = shuffled_indices_dataset[size_of_test_set:]
    return dataset.iloc[indices_of_train_set], dataset.iloc[indices_of_test_set]

In [ ]:
#Executing the function to split the dataset
    #Assumption: 20% of data will be for the test set (ratio =0.2)
    #The newly created and returned index will help build the training set and test set.
    
train_set2, test_set2 = split_dataset(training_data, 0.2)

#Printing the "len" of the new training set (version 2)
len(train_set2)

In [ ]:
#Printing the "len" of the new test set (version 2)
len(test_set2)

In [ ]:
#Printing a couple of rows of the new training set (version 2)
train_set2.head(5)

In [ ]:
#Printing a couple of rows of the new test set (version 2)
test_set2.head(5)

## __3.2. Option 2: Using the Scikit Learn function to split data randomly__

In [ ]:
#Executing the function to split the dataset
    #Assumption: 20% of data will be for the test set (ratio =0.2)
    #In this function, shuffle by default is True.
    #Save plenty of time!

train_set3, test_set3 = train_test_split(training_data, test_size = 0.2, random_state = 66)

#Printing the "len" of the new training set (version 3)
len(train_set3)

In [ ]:
#Printing the "len" of the new test set (version 3)
len(test_set3)

In [ ]:
#Printing a couple of rows of the new training set (version 3)
train_set3.head(5)

In [ ]:
#Printing a couple of rows of the new test set (version 3)
test_set3.head(5)

# __4. Visualize the Data__

This process is helpful to gain insights about the training data.
> While large training datasets have to be sampled for the visualization stage, small training datasets like ours (891rows of data) can be entirely explored.

In [ ]:
#In this scenario, the original "training set" given by this Kaggle problem is used

#Dataframe.plot

#Some relevant arguments of "Dataframe.plot" function are the followings:
    #s: The marker size (radius of each circle of the scatterplot. In this case, the bigger the circle, the bigger the Fare.
    #c: The marker colors (represents if the passenger survived or not)
    #alpha: The alpha blending value, between 0 (transparent) and 1 (opaque). This argument allows us to see the intersection between data points of the scatterplot.
    #colorbar: This bar works like a guide for the user. It is a very helpful complement to the "c" and "cmap" arguments.
    #cmap: The colormap. There are several ways to deal with this argument.
        #Option 1:
            #Using the function "mpl.colors.ListedColormap([,])"
            #The function "mpl.colors.ListedColormap([,])" needs to load the following:
                #Library: matplotlib
                #Alias: mpl
        #Option 2: Preferred!
            #Using the function "matplotlib.pyplot.get_cmap(name=None, lut=None)"
            #The function "get_cmap" uses information previously loaded, such as:
                #Library: matplotlib
                #Module: pyplot
                #Alias: plt
        #To know more about the function "get_cmap", see the following markdown
training_data.plot(kind= "scatter", x="Fare", y="Pclass", s=2*training_data["Fare"], figsize = (8,8), c="Survived", cmap= plt.get_cmap("cividis"), label = "Fare", colorbar=True ,alpha= 0.2)

> Detailed information about how the function "get_cmap" works (arguments, etc.)

<a href="https://matplotlib.org/stable/api/cm_api.html#module-matplotlib.cm">Matplotlib link about Return a color map specified through cmap.</a>


> Detailed information about the argument "colormap" chosen: Sequential (to be specific: "cividis")

<a href="https://matplotlib.org/stable/gallery/color/colormap_reference.html">Matplotlib link about Colormap Reference</a>

<a href="https://matplotlib.org/stable/tutorials/colors/colormaps.html">Matplotlib link about Choosing Colormaps in Matplotlib</a>

## __4.1. Looking for correlations (in numerical attributes)__

### __4.1.1. Correlation(Standard Correlation Coefficient)__

This process is helpful to gain insights about the training data.

1. __"Compute the Standard Correlation Coefficient" (i.e., Pearson's Correlation Coefficient {r})__ between the numerical attributes of our dataset. When using the Standard Correlation Coefficient, it is important to be aware of the following points:

> It only measures "linear correlation".

> It ranges from -1 to 1.

> The values of r=1 and r=-1 show a strong linear correlation between the variables.

> A negative linear correlation, indicated by a correlation coefficient ranging from 0 to -1, means that as one attribute increases, the other attribute tends to decrease.

> A positive linear correlation, indicated by a correlation coefficient ranging from 0 to 1, means that as one attribute increases, the other attribute tends to increase as well."

> The value of r=0 shows a no linear correlation between the variables.

2. __Options to "Compute the Standard Correlation Coefficient"__:

> __Option 1__: Compute the "Standard Correlation Coefficient" of all numerical attributes of our dataset between each other.

> __Option 2 (more efficient way)__: "Compute the Standard Correlation Coefficient" of all numerical attributes of our dataset against an interesting attribute (i.e. the "target value" that we want to predict using the Test Set).

In [ ]:
#Applying Option 1 (Compute the "Standard Correlation Coefficient" of all numerical attributes of our dataset between each other)

    #Interesting outputs (correlations): "Survived vs Pclass", "Survived vs Fare","Pclass vs Fare","Pclass vs Age", "Age vs SipSp", "SibSp vs Parch", "Parch vs Age", and Parch vs Fare".
correlation_matrix = training_data.corr()

correlation_matrix

In [ ]:
#Applying Option 2 ("Compute the Standard Correlation Coefficient" of specific numerical attributes of our dataset against an interesting attribute)

    #Interesting outputs (correlations): "Survived vs Pclass" and "Survived vs Fare"

correlation_matrix["Survived"].sort_values(ascending = False)

### __4.1.2. Correlation (using graphs)__

This process is helpful to gain insights about the training data.

1. "__Graphical Correlation between the numerical attributes__" of our dataset.  When it is used, could be wortwhile to be aware of the followings points:

> If you have a large number of numerical attributes, select only the most promising ones for this process, using as a guide the outputs of section "__4.1.1. (Correlation - Standard Correlation Coefficient)__". Otherwise we are going to get a graphical mess (do not forget that "the number of plots" = "number of attributes squared").

> If we are dealing with attributes that have tail-heavy distribution {according to the section "__2.1.5. Summary of numerical attributes (in a graph way)__"}, transform those attributes before the "Graphical Correlation Procedure" could be helpful.

> Quirky data points can be spotted during the process of "Graphical Correlation". Clean quirky data points after the process of "Graphical Correlation" might be helpful for the performance of the model during the test phase.

In [ ]:
#Graphical Correlation between all the numerical attributes of our trainig set
scatter_matrix(training_data, figsize=(8,8))

In [ ]:
#Plotting the correlation between the most promising numerical attributes of our training set.
interest_attributes = ["Survived", "Pclass", "Fare", "Age"]

scatter_matrix(training_data[interest_attributes], figsize=(8,8))

<div class="alert alert-block alert-danger"><b>Important:</b> In the previous steps, we have applied several methods or techniques to gain insights from the data.

Beyond this point, we will begin the process of modifying, transforming, changing, or editing the training data in order to train the model and obtain the best possible model. </div>

<div class="alert alert-block alert-warning"><b>Important:</b> Data Traceability </div>

Name of dataframe | Original | 
---|---|
training_data | Original Trainind Dataset | 

## __4.2. Attribute Combinations__

Basically, this is the last step in the process of gain insights about the training data.

We can create "new attributes" from the combinations of the most promising existing attributes. The "new attributes" could came from the helpful insights that we have seen in the section "__4.1. Looking for correlations (in numerical attributes)__".
> If we want to know how good are those "new attributes", we can compute the __"Standard Correlation Coefficient" (i.e., Pearson's Correlation Coefficient {r})__ against an interesting attribute (i.e. the "target value" that we want to predict using the Test Set).

According to the performance of the model in the Test Set we can assses if it is wortwhile to use new attribute combinations (i.e., it is an iterative process.)

In [ ]:
#Training dataset (original dataframe)
training_data

In [ ]:
#Highly recommended: Before applying "attribute combination" do a copy of the original training dataset.
#We are agoing to add some columns to original dataframe; therefore a dataframes's copy is very useful.

training_data2 = training_data.copy()
training_data2

In [ ]:
#The new attributes (that will be added as columns to the training set {i.e. added to existing dataframe}) are the followings:
    #class per fare
    #number of parents / children per number of siblings / spouses

training_data2["class per fare"]= training_data2["Pclass"]/training_data2["Fare"]
training_data2["number of parents / children per number of siblings / spouses"]= training_data2["SibSp"]/training_data2["Parch"]

In [ ]:
#We can see interesting outputs (i.e., correlations) between the target value "Survived" and the newly created attributes ("class per fare", and "number of parents / children per number of siblings / spouses")

correlation_matrix2 = training_data2.corr()
correlation_matrix2["Survived"].sort_values(ascending = False)

In [ ]:
#General information or description of the "old and newly created attributes". 
training_data2.info()

In [ ]:
#Plotting the correlation between the newly attribute created and the target value (Survived)

    #The newly created attribute in the plot is "number of parents / children per number of siblings / spouses"

training_data2.plot(kind="scatter", x="number of parents / children per number of siblings / spouses", y="Survived", s=500*training_data2["class per fare"], figsize = (6,6), alpha=0.2)

In [ ]:
#Plotting the correlation between the newly attribute created and the target value (Survived)

    #The newly created attribute in the plot is "class per fare"

training_data2.plot(kind="scatter", x="class per fare", y="Survived", s=500*training_data2["class per fare"], figsize = (6,6), alpha=0.2)

In [ ]:
#New training set dataframe (including the new columns, i.e. the newly created attributes).
training_data2

# __5. Preparing the data__

## __5.1. Data Cleaning__

From section "__2.1.1. Getting information/description of data__" we know which attributes have NA values. To be specific they are the followings:
> Age (numerical attribute)

> Cabin

> Embarked

Four options to deal with NA values in "Numerical Attributes" are the followings:
1. __Option 1__: The dropna() method removes the rows that contains null values. 

> The dropna() method returns a new dataframe object unless the "inplace" parameter is set to True, in that case the dropna() method does the removing in the original datadrame instead.

> The "subset" parameter specifies where (column) to look for NULL values.

2. __Option 2__: The "drop()" function is a drastic method in constrast with "dropna().

> The first argument are the labels or indexes to drop. If more than one, specify them in a list. 

> The "axis" parameter: allow us to establish which axis to check (1 for columns or 0 for rows).

3. __Option 3__: The "fillna()" method replaces the null values with a specified value. 

<div class="alert alert-block alert-warning"><b>Important:</b> Data Traceability </div>

Name of dataframe | Original | Transformation Applied N°1 | 
---|---|---|
training_data | Original Trainind Dataset | 
training_data2 | Original Trainind Dataset | Attribute Combination |

In [ ]:
#Option 1
training_data21 = training_data2.copy()

training_data21.dropna(subset=["Age"], inplace=True)

training_data21

In [ ]:
#Option 2
training_data22 = training_data2.copy()

training_data22.drop("Age", axis=1, inplace=True)

training_data22

In [ ]:
#Option 3
training_data23 = training_data2.copy()

#We choose to fill NaN values with the "median value"of the attribute "Age"
medianvalue = training_data23["Age"].median()
training_data23["Age"].fillna(medianvalue, inplace=True)

training_data23

## __5.2. Dealing with Text and Categorical Attributes__

### __5.2.1. Encoding__

Until now we have seen only how to get insights of "numerical attributes", In the following lines we are going to see how to deal with "text and categorical attributes" considering that most ML algorithms use to work better with numeric values only. 

Based in what we described in the previous lines, we are going to transform or encode" the text or categories into numbers. Just to check:

> The "categorical attributes" in our training data set are: "Sex" and "Embarked".

> The "text attributes" in our training data set are: "Name", "Ticket", "Cabin".

This process works very well when attributes has a limited number of possible categories.

### __5.2.1.1. Encoding (Feature "Sex")__

In [ ]:
#Working with the "categorical attributes"

#Printing the categorical attribute "Sex" as a dataframe.
training_data_cat1 = training_data21[["Sex"]]
training_data_cat1

In [ ]:
#The function "OrdinalEncoder()" encode categorical features as an integer array (0 to n_categories - 1).

ordinal_encoder1 = OrdinalEncoder()
training_data_cat1_encoded = ordinal_encoder1.fit_transform(training_data_cat1)
training_data_cat1_encoded[0:10]

In [ ]:
#Getting the list of categories of the feature "Sex" (before encoding.) using the "categories_" attribute.

    #It is considered a "fitted parameter" (also called "learned parameter")

    #The columns in the array are returned alphabetically (in this case "female, male").

ordinal_encoder1.categories_

In [ ]:
#More information about the list of categories of the feature "Sex".To be specific the length and type of the list and elements of the list.

inf1 = type(ordinal_encoder1.categories_)
print(inf1)
inf2 = type(ordinal_encoder1.categories_[0])
print(inf2)

inf3 = len(ordinal_encoder1.categories_)
print(inf3)
inf4 = len(ordinal_encoder1.categories_[0])
print(inf4)

### __5.2.1.2. Encoding (Feature "Embarked")__

In [ ]:
#Printing the categorical attribute"Embarked" as a dataframe.

training_data_cat2 = training_data21[["Embarked"]]
training_data_cat2

In [ ]:
#The function "OrdinalEncoder()" encode categorical features as an integer array (0 to n_categories - 1).

ordinal_encoder2 = OrdinalEncoder()
training_data_cat2_encoded = ordinal_encoder2.fit_transform(training_data_cat2)
training_data_cat2_encoded[0:10]

In [ ]:
#Getting the list of categories of the feature "Embarked" (before encoding.) using the "categories_" attribute.

    #It is considered a "fitted parameter" (also called "learned parameter")

    #The columns in the array are returned alphabetically (in this case "C", "Q", "S" and nan").

ordinal_encoder2.categories_

### __5.2.2. Quick review about some relevant commands and concepts from Scikit-Learn__

Scikit learn is an amazing ML toolkit for Python language.

1. __Estimators__

> Its main goal is to estimate some parameters based on the dataset.

> An estimator is an object that can learn from data and make predictions. 

> All estimators have a "fit" method (also called the "training method" or "learning method").

> The function "fit()" trains or fits the algorithm on the training data with the aim of estimate the "parameters" or learning "model's parameters", also known the "weights" (i.e. the coefficients located alongside the features).

> The arguments of the function "fit()" are the "features of the training dataset", and the "target values of the training dataset" (in the case of Supervised ML). 

> Fitted parameters (also called "learned parameters") use an underscore " _ " as a suffix.

2. __Transformers__

> The method "transform()" allow us to modify the data, such as scaling, encoding, imputing, or extracting features. We can also create our own custom transformers by implementing the "fit" and "transform" methods. 

> It can make your code more readable, reusable, and robust. They can also help you avoid data leakage and perform cross-validation more easily.

> The argument of the function "transform()" is the dataset. The output is the transformed dataset. 

> The method "fit_transform()" is equivalent to calling the functions "fit()" and "transform()". This is a convenient and efficient method for modelling (i.e., estimates parameters or learns model parameters) and transforming the training data simultaneously using the respective "learned parameters".

> If we use the "fit()" function on the test data, we will compute new parameters and will let our model learn about the test data (i.e., it will be a model biased towards particular features or values of the test data). Therefore, we do not use the "fit()" method on the test data.


3. __Predictors__

> It has the "predict()" method which allow us to to make predictions. This function takes advantage of the fitted parameters (also called "learned parameters") got by the method "fit()".

> The argument of the function "predict()" is usually the "features of the test dataset".

4. __Pipeline__

> It is a way of chaining a sequence of steps that can include data "transformations" and a "final estimator". 

> The multiple steps that can chain the pipeline object are any "estimators" that implement the "fit" and "transform" methods. The last step of the "pipeline" must be an "estimator" that implements the "fit" method.

> The "pipeline" object simplifies the code by avoiding repeated calls to "transform" and "fit" methods. 

> "make_pipeline" function from scikit-learn help us to create a pipeline object that can chain multiple steps.

5. __Classes__

> The "classes" from scikit-learn python are not the same as the "estimators" or "transformers". 

>The "classes" are the Python data structures that define the behavior and attributes of the "objects" such as "estimators" and "transformers". Those "objects" that are created from the "classes". 

<div class="alert alert-block alert-warning"><b>Important:</b> Datasets in scikit-learn are represented as "Numpy arrays" or "Scipy sparse matrices". In other words, when we use scikit-learn we are going to get the output of our codes in any of these two data types formats. </div>

According to scikit-learn website (<a href="">https://scikit-learn.org/stable/faq.html </a>), we should consider the following:
>"_The homogeneous NumPy and SciPy data objects currently expected are most efficient to process for most operations_"

>"_Restricting input to homogeneous types therefore reduces maintenance cost and encourages usage of efficient data structures._"

>"_Most of scikit-learn assumes data is in NumPy arrays or SciPy sparse matrices of a single numeric dtype. These do not explicitly represent categorical variables at present. Thus, unlike R’s data.frames or pandas.DataFrame, we require explicit conversion of categorical features to numeric values,..._"

### __5.2.3. One-Hot Encoding__

The previous "encoding procedure" is simple and useful; however, it has the following pitfall: ML algorithms may consider that encoded categories next to each other are similar. 

> It could be a problem if the "categorial attributes" are relevant features for our model and, if the categories of those "categorial attributes" are not ordered.

Considering what is described previously, the recommended method for encoding is "One-Hot Encoding".

> "One-Hot Encoding" creates a binary system (with the values 0 or 1) per category. If the category is chosen, then it will have the value 1 (one); therefore, all the other categories will have the value 0 (zero).

> 1 (one) is hot and 0 (zero) is cold.

> The output of the "One-Hot Encoding" procedure is a huge matrix full of zeros and just a couple of ones (per category). This type of matrix is known as "Sparse Matrix".

#### __5.2.3.1. One-Hot Encoding (Feature "Sex")__

In [ ]:
#Applying "One-Hot Encoding" to the feature "Sex"

    #The output was a huge matrix (891 rows x 2 columns) full of zeros and just a couple of ones.

    #Types of sparse matrices in Scipy: CSR(Compressed Sparse Row) and CSC(Compressed Sparse Column).

    #Columns for the category of "male" and "female".

OneHot_encoder1 = OneHotEncoder()
training_data_cat1_OneHotencoded = OneHot_encoder1.fit_transform(training_data_cat1)
training_data_cat1_OneHotencoded

In [ ]:
#Transforming the "Scipy Sparse Matrix" {CSR(Compressed Sparse Row)} into dense Numpy Array

    #In essence, the output is an array with the same shape and containing the same data represented by the sparse matrix.

    #Output = 2 columns of binary values in the array because there are only 2 categories). 

    #The columns in the array are returned alphabetically (in this case "female, male").

training_data_cat1_OneHotencoded.toarray()

#### __5.2.3.2. One-Hot Encoding (Feature "Embarked")__

In [ ]:
#Applying "One-Hot Encoding" to the feature "Embarked"

    #The output was a huge matrix (891 rows x 4 columns) full of zeros and just a couple of ones.

    #Columns for the category of "S", "Q", "C" and nan...

OneHot_encoder2 = OneHotEncoder()
training_data_cat2_OneHotencoded = OneHot_encoder2.fit_transform(training_data_cat2)
training_data_cat2_OneHotencoded

In [ ]:
#Transforming the "Scipy Sparse Matrix" {CSR(Compressed Sparse Row)} into dense Numpy Array.

    #In essence, the output is an array with the same shape and containing the same data represented by the sparse matrix.

    #Output = 4 columns of binary values in the array because there are only 4 categories). 

    #The columns in the array are returned alphabetically (in this case "C", "Q", "S" and nan").

training_data_cat2_OneHotencoded.toarray()

<div class="alert alert-block alert-warning"><b>Important:</b> Data Traceability </div>

Name of dataframe | Original | Transformation Applied N°1 | Transformation Applied N°2 |
---|---|---|---|
training_data | Original Trainind Dataset | 
training_data2 | Original Trainind Dataset | Attribute Combination |
training_data21 | Original Trainind Dataset | Attribute Combination | Data Cleaning |

#### __5.2.3.3. One-Hot Encoding (continue - Feature "Sex")__

In [ ]:
#For the following steps, we are going to choose only the outputs of the process of..
#..."One-Hot Encoding (Feature "Sex")" because based in the previous preprocessing outputs, it...
#...could be an important feature.

    #Creating a copy of the dataframe that only has the feature "Sex".

training_data_cat11 = training_data_cat1.copy()
training_data_cat11

In [ ]:
#We put together the "categories of the encoder" (to be specific: male and female) from...
#...step 5.2.1.1. Encoding (Feature "Sex") and the "dense Numpy Array" from...
#...step 5.2.3.1. One-Hot Encoding (Feature "Sex")

training_data_cat11[ordinal_encoder1.categories_[0]]=training_data_cat1_OneHotencoded.toarray()
print(training_data_cat11)

#In addition, we print the type of the ouptput (i.e. a dataframe).

print(type(training_data_cat11))

In [ ]:
#Dropping the column "Sex" from the previous dataframe. I do this because in the following step we are...
#...going to concatenate this small dataframe (output from the "One Hot Encoding Activities") and the...
#...main training data transformed. Therefore, taking into account that both have the column "Sex",..
#...I prefer to avoid misunderstanding about having 2 columns with the same name and values.

training_data_cat11.drop(training_data_cat11.columns[0], axis=1, inplace=True)

training_data_cat11

In [ ]:
#Concatenate dataframes (output from the "One Hot Encoding Activities" and the "main training data transformed")

training_data_cat211 = pd.concat([training_data21, training_data_cat11], axis=1)
training_data_cat211

## __5.3. Feature Scaling__

__Key ideas__:

1. Many ML algorithms are highly sensitive or do not work well when the features of the model (independent variables) have different scales (for instance: a feature has values between 3 and 7, other feature has values between 400 and 1000 and another feature of the same model has values between 10000 and 500000). Therefore, applying "Feature Scaling" (i.e. a transformation) allows us to normalize the features and makes ML problems easy to address by ML algorithms.

> Based in what we see in the section __2.1.4. Summary of numerical attributes__ we considered apply "feature scaling"

> Based in what we see in the section __2.1.5. Summary of numerical attributes (in a graph way)__ we considered apply "feature scaling"

2. It is recommended to apply the "Feature Scaling" procedure after the processes of "Attribute Combination" and "Encoding Categorial/Text Attributes".

3. ML algorithms (for instance: linear regression, logistic regression, PCA, ANN,...) that use "Gradient Descent" as an optimization technique should be checked with "Feature Scaling" because in that way we are able to guarantee a regular size steps during the gradient descent process and properly convergence.

4. Distance-based ML algorithms (for instance: SVM, KNN, K-means...) should be checked with "Feture Scaling" because they use distance between data points to show similarity between them.

5. There are several ways to apply "Feature Scaling"; however, we are going to see the following two:

5.1. __Normalization (also called "Min - Max Scaling")__

> It adjusts the values of the features (numeric features) to end up ranging between 0 and 1.

> It is sensitive to outliers points, because Normalization formula works with maximun and minimun values of the feature.

> It is useful when the features have data distributions that do not follow a Gaussian Distribution or when there is no assumption about the data distributions of features.

> It can be useful in "Encoded Categorial/Text Attributes" (One-Hot Encoding).

5.2. __Standardization (also called "Z-Score Normalization")__ 
<-- <span style="color:black">__This option is NOT DONE in this Notebook.__</span> 

> It adjusts the values of the features (numeric features) to end up ranging between no specific values. It standardize values to have mean value equal to 0 and standard deviation of 1.

> It is not sensitive to outliers in the data, because Standardization formula works with the mean and the standard deviation value of each the feature.

> It is useful when the features have data distributions that follow a Gaussian Distribution.

6. Feature scaling can affect both the "features" and the "parameters" of our model, depending on the type of scaling and the type of model (linear models, non-linear models, ...)

7. Despite the conditions described previously, I would recommend train the ML algorithm using the "raw training data", "normalize training data", and "standardize training data". Then, check and compare the peformance of each model with error analysis.

<div class="alert alert-block alert-warning"><b>Important:</b> Data Traceability </div>

Name of dataframe | Original | Transformation N°1 | Transformation N°2 | Transformation N°3
---|---|---|---|---|
training_data | Original Training Dataset | 
training_data2 | Original Training Dataset | Attribute Combination |
training_data21 | Original Training Dataset | Attribute Combination | Data Cleaning |
training_data_cat211 | Original Training Dataset | Attribute Combination | Data Cleaning | Categorical Attributes

### __5.3.1. Normalization (also called "Min - Max Scaling") on Training Dataset__

In [ ]:
#Normalization (Training Dataset) - Part 1

    #Making a copy of the last version of the training set.
training_data_norm = training_data_cat211.copy()

In [ ]:
#Normalization (Training Dataset) - Part 2

    #Choosing only the numerical independent features (include those that came from "attribute distribution").
training_data_norm = training_data_norm[["Pclass","Age", "SibSp", "Parch", "Fare", "class per fare","female", "male"]]

In [ ]:
#Normalization (Training Dataset) - Part 3.1

#Addressing infinity values. 
#I did this because infinity values do not allow the "Normalization Feature Scaling" function properly works.

    #Checking if there are cells inside the dataframe with finite values. In this case, due to huge amount of data that we have it is hard to see the "False" items.
    #"False" = infinity values
    #"True" = finite values
infinityrow_norm = np.isfinite(training_data_norm)
infinityrow_norm

In [ ]:
#Normalization (Training Dataset) - Part 3.2

    #Counting the cells that have infinity values
    #In this case the output is: 7 cells with infinity values
quantity_norm = np.isinf(training_data_norm).values.sum()
print(quantity_norm)

In [ ]:
#Normalization (Training Dataset) - Part 3.3

    #Discovering which columns have cells with infinity values 
    #In this case the output is: the column or feature "class per fare".
    #We do this because the detection of an "infinity value" was difficult in the section "Normalization - Part 3.1" 
col_norm = training_data_norm.columns.to_series()[np.isinf(training_data_norm).any()]
print(col_norm)

In [ ]:
#Normalization (Training Dataset) - Part 3.4

    #Replacing the infinity values with NaN
    #In this step we still have 714 data rows, just like in the sections:
        #"Normalization - Part 1
        #"Normalization - Part 2
        #"Normalization - Part 3.1
        #"Normalization - Part 3.2
        #"Normalization - Part 3.3
training_data_norm.replace([np.inf, -np.inf], np.nan, inplace=True)
training_data_norm

In [ ]:
#Normalization (Training Dataset) - Part 3.5

    #Dropping all the rows with NaN values
    #In this step we have a new quantity of rows (to be specific: 707 rows) because we dropped rows with NaN values.
training_data_norm.dropna(inplace=True)
training_data_norm

In [ ]:
#Normalization (Training Dataset) - Part 4

    #Fitting the training data (i.e. learning parameters from the trainin data) with the "Normalization Feature Scaling"
fit_data_norm = MinMaxScaler().fit(training_data_norm)

    #Transforming and printing the training data
training_data_norm2 = fit_data_norm.transform(training_data_norm)
print(training_data_norm2)

    #Printing the datatype of the normalized training set.
print(type(training_data_norm2))

    #Printing the shape (dimensions) of the normalized training set.
print(training_data_norm2.shape)

In [ ]:
#Normalization (Training Dataset) - Part 5.1

    #BoxPlot of the Training Set (Raw Numeric Features)

    #This is the same box plot of section "2.1.5. Summary of numerical attributes (in a graph way)"

boxplot = training_data.boxplot(column=['Pclass',"Age", "SibSp", "Parch", "Fare"],figsize = (7,7)) 
plt.show

In [ ]:
#Normalization (Training Dataset) - Part 5.2

    #BoxPlot of the Training Set (Normalize Features)

    #We can see an dramatic difference between the previous and the current boxplot (i.e., before and after the "Normalization" Process).

fig, ax = plt.subplots()
ax.boxplot(training_data_norm2) 
ax.set_xticklabels(["Pclass","Age", "SibSp", "Parch", "Fare", "class per fare","female", "male"],rotation=90)
plt.show

In [ ]:
#Normalization (Training Dataset) - Part 6

    #Convert the NumPy Array to Pandas DataFrame
    #This Numpy Array came from "Normalization (Training Dataset) - Part 4"
    #The main goal is to convert the newly created numpy array (also called "training_data_norm2") to a dataframe using the index of the training set from section "Normalization (Training Dataset) - Part 3.5"

training_data_norm21 = pd.DataFrame(training_data_norm2, columns = ["Pclass","Age", "SibSp", "Parch", "Fare", "class per fare","female", "male"],index = training_data_norm.index)

print(training_data_norm21)
print(type(training_data_norm21))

# __6. Train the Model__

<div class="alert alert-block alert-warning"><b>Important:</b> Data Traceability </div>

Name of dataframe | Original | Transformation N°1 | Transformation N°2 | Transformation N°3 | Transformation N°4
---|---|---|---|---|---|
training_data | Original Training Dataset | 
training_data2 | Original Training Dataset | Attribute Combination |
training_data21 | Original Training Dataset | Attribute Combination | Data Cleaning |
training_data_cat211 | Original Training Dataset | Attribute Combination | Data Cleaning | Categorical Attributes
training_data_norm21 | Original Training Dataset | Attribute Combination | Data Cleaning | Categorical Attributes | Normalization

## __6.1. Training Dataset__

In [ ]:
#"Final training set" after several transformation (described in the previous chart)

training_data_norm21

In [ ]:
#Getting the "target value" from the training set build in the last step of "One-Hot Encoding"

training_data_targetY = training_data_cat211["Survived"]
training_data_targetY

In [ ]:
#Joining the independent features (numeric features) called "training_data_norm21" and the target values (responses) called "training_data_cat211" of the training set.

#I do this joining because the length of both named dataframes is different and I want to match their indexes
    #the dataframe "training_data_norm21" has 707 rows
    #the dataframe "training_data_cat211" has 714 rows

training_x_y_values = training_data_norm21.join(training_data_cat211["Survived"],how="inner",rsuffix='_extra')
training_x_y_values

In [ ]:
#Splitting the training set ("independent features" from the "target value") 

    #Getting the "target value" from the training set build in the previous step.

training_data_targetY = training_x_y_values["Survived"]
training_data_targetY

In [ ]:
#Dropping the "target column" from the training set.

    #Getting the "independent features" (only the numeric) from the training set.

training_data_newX = training_x_y_values.drop(["Survived"], axis='columns')
training_data_newX

## __6.2. Test Dataset__

The same transformation that the "Training Dataset" underwent in the previous steps will be applied to the "Test Dataset". Just to remind you, these transformations are as follows:

> Data cleaning (dropping NA values)

> Attribute Combination

> Enconding categorical attributes (objects)

> Feature normalization

In [ ]:
#Importing the original Test Set

official_test_data = pd.read_csv("/kaggle/input/titanic/test.csv")

    #Doing a copy of the original Test Set
test_data1 = official_test_data.copy()

### __6.2.1. Test Dataset - Getting information/description of data__

In [ ]:
test_data1.info()

In [ ]:
test_data1.describe()

### __6.2.2. Test Dataset - Data Cleaning__

From section "2.1.1. Getting information/description of data" we know which attributes have NA values. To be specific they are the followings:
> Age (numerical attribute)

> Fare (numerical attribute)

> Cabin

In [ ]:
#Cleaning (Test Dataset)

    #Replacing all the NaN values with specific values (i.e. the mean value of every column in which there are NaN values).
    
    #In this step we end with the same quantity of rows (to be specific: 418 rows) because we did not drop the rows with NaN values (i.e. we filled with specific values).
    
    #inplace: If True, fill in-place.

test_data12 = test_data1.copy()
    
specific_values = {"Age": test_data12['Age'].mean(), "Fare": test_data12['Fare'].mean()}

test_data12.fillna(value = specific_values, inplace=True)

test_data12

In [ ]:
#Just to check that every NaN value if features "Age" and "Fare" disappeared.

test_data12.info()

### __6.2.3. Test Dataset - Attribute Combination__

In [ ]:
#The new attribute (that will be added as a column to the test set {i.e. added to existing dataframe}) is the following:
    #class per fare

test_data2 = test_data12.copy()
    
    #New test set dataframe (including the new column, i.e. the newly created attribute).
test_data2["class per fare"]= test_data2["Pclass"]/test_data2["Fare"]
test_data2

In [ ]:
#Just to check the description of the dataframe considering the feature "Class per Fare".

test_data2.describe()

In [ ]:
#Just to check that every NaN value of features "Class per Fare" disappeared.

test_data2.info()

### __6.2.4. Test Dataset - One-Hot Encoding (Feature "Sex")__

In [ ]:
#Working with the "categorical attributes"

    #Printing the categorical attribute "Sex" as a dataframe.
test_data_sex = test_data2[["Sex"]]
test_data_sex

In [ ]:
#Applying "One-Hot Encoding" to the feature "Sex"

    #The output was a huge matrix (331 rows x 2 columns) full of zeros and just a couple of ones.

    #Types of sparse matrices in Scipy: CSR(Compressed Sparse Row) and CSC(Compressed Sparse Column).

    #Columns for the category of "male" and "female".

OneHot_encoder1 = OneHotEncoder()
test_data_sex_OneHotencoded = OneHot_encoder1.fit_transform(test_data_sex)
test_data_sex_OneHotencoded

In [ ]:
#Transforming the "Scipy Sparse Matrix" {CSR(Compressed Sparse Row)} into dense Numpy Array

    #In essence, the output is an array with the same shape and containing the same data represented by the sparse matrix.

    #Output = 2 columns of binary values in the array because there are only 2 categories). 

    #The columns in the array are returned alphabetically (in this case "female, male").

test_data_sex_OneHotencoded.toarray()

In [ ]:
#For the following steps, we are going to choose only the outputs of the process of..
#..."One-Hot Encoding (Feature "Sex")" because based in the previous preprocessing outputs, it...
#...could be an important feature.

    #Creating a copy of the dataframe that only has the feature "Sex".

test_data_sex2 = test_data_sex.copy()

In [ ]:
#We put together the "categories of the encoder" (to be specific: male and female) from...
#...and the "dense Numpy Array"

test_data_sex2[ordinal_encoder1.categories_[0]]=test_data_sex_OneHotencoded.toarray()
print(test_data_sex2)

    #In addition, we print the type of the ouptput (i.e. a dataframe).

print(type(test_data_sex2))

In [ ]:
#Dropping the column "Sex" from the previous dataframe. I do this because in the following step we are...
#...going to concatenate this small dataframe (output from the "One Hot Encoding Activities") and the...
#...main training data transformed. Therefore, taking into account that both have the column "Sex",..
#...I prefer to avoid misunderstanding about having 2 columns with the same name and values.

test_data_sex2.drop(test_data_sex2.columns[0], axis=1, inplace=True)

test_data_sex2

In [ ]:
#Concatenate dataframes (output from the "One Hot Encoding Activities" and the "main training data transformed")

test_data3 = pd.concat([test_data2, test_data_sex2], axis=1)
test_data3

### __6.2.5. Test Dataset - Feature Scaling (Normalization)__

In [ ]:
#Normalization (Test Dataset) - Part 1

    #Making a copy of the last version of the Test set.
test_data4 = test_data3.copy()

In [ ]:
#Normalization (Test Dataset) - Part 2

    #Choosing only the numerical independent features (include those that came from "attribute distribution").
test_data4 = test_data4[["Pclass","Age", "SibSp", "Parch", "Fare", "class per fare","female", "male"]]

In [ ]:
#Normalization (Test Dataset) - Part 3.1

#Addressing infinity values. 
#I did this because infinity values do not allow the "Normalization Feature Scaling" function properly works.

    #Checking if there are cells inside the dataframe with finite values. In this case, due to huge amount of data that we have it is hard to see the "False" items.
    #"False" = infinity values
    #"True" = finite values
infinityrow_norm_test = np.isfinite(test_data4)
infinityrow_norm_test

In [ ]:
#Normalization (Test Dataset) - Part 3.2

    #Counting the cells that have infinity values
    #In this case the output is: 2 cells with infinity value
quantity_norm_test = np.isinf(test_data4).values.sum()
quantity_norm_test

In [ ]:
#Normalization (Test Dataset) - Part 3.3

    #Discovering which columns have cells with infinity values 
    #In this case the output is: the column or feature "class per fare".
    #We do this because the detection of an "infinity value" was difficult in the section "Normalization - Part 3.1" 
col_norm_test = test_data4.columns.to_series()[np.isinf(test_data4).any()]
col_norm_test

In [ ]:
#Normalization (Test Dataset) - Part 3.4

    #Replacing the infinity values with NaN
    #In this step we still have 418 data rows, just like in the sections:
        #"Normalization - Part 1
        #"Normalization - Part 2
        #"Normalization - Part 3.1
        #"Normalization - Part 3.2
        #"Normalization - Part 3.3
test_data4.replace([np.inf, -np.inf], np.nan, inplace=True)
test_data4

In [ ]:
#Normalization (Test Dataset) - Part 3.5

    #In this section we tranform the "NaN values" of the previous section "Normalization (Test Dataset) - Part 3.4" into another values (to be specific the "mean value" of the column "class per fare")

test_data41 = test_data4.copy()

specific_values2 = {"class per fare": test_data41['class per fare'].mean()}

test_data41.fillna(value = specific_values2, inplace=True)

test_data41

In [ ]:
#Normalization (Test Dataset) - Part 4

#Transforming and printing the Test data
test_data_norm = fit_data_norm.transform(test_data41)
test_data_norm

In [ ]:
#Printing the datatype of the normalized Test set.
type(test_data_norm)

In [ ]:
#Printing the shape (dimensions) of the normalized Test set.
test_data_norm.shape

In [ ]:
#Normalization (Test Dataset) - Part 5.1

    #BoxPlot of the Test Set (Raw Numeric Features)

    #This is the same box plot of section "2.1.4. Summary of numerical attributes (in a graph way)"

boxplot = test_data3.boxplot(column=['Pclass',"Age", "SibSp", "Parch", "Fare"],figsize = (7,7)) 
plt.show

In [ ]:
#Normalization (Test Dataset) - Part 5.2

    #BoxPlot of the Test Set (Normalize Features)

    #We can see an dramatic difference between the previous and the current boxplot (i.e., before and after the Normalization Process).

fig, ax = plt.subplots()
ax.boxplot(test_data_norm) 
ax.set_xticklabels(["Pclass","Age", "SibSp", "Parch", "Fare", "class per fare","female", "male"],rotation=90)
plt.show

In [ ]:
#Normalization (Test Dataset) - Part 6

    #Convert the NumPy Array to Pandas DataFrame
    #This Numpy Array came from "Normalization (Test Dataset) - Part 4"
    #The main goal is to convert the newly created numpy array (also called "test_data_norm") to a dataframe using the index of the test set from section "Normalization (Test Dataset) - Part 3.5"

test_data_norm2 = pd.DataFrame(test_data_norm, columns = ["Pclass","Age", "SibSp", "Parch", "Fare", "class per fare","female", "male"],index = test_data4.index)

print(test_data_norm2)
print(type(test_data_norm2))

## __6.3. Training and doing a small evaluation (on the Training Dataset)__

The "Linear Regression Method" to find the optimal coefficients that minimize the residual sum of squares applied the following techniques:

1.  __Normal Equation__:

> It is an analytical solution that does not require iteration (i.e. it is a "closed form" solution). 

> We don't have to set initial values for the coefficients, because the "closed-form equation" that does not depend on the initial values.

> The Normal Equation is computed using the standard matrix factorization technique that can solve linear systems efficiently called "Singular value decomposition" of X (where "X" is the array the features of the training data).

> Complexity: This method computes "inverse", therefore it has a computational complexity of (number of features)^2 to (number of features)^3. About the number of instances, it has a "linear complexity" (in essence, this method can handle datasets that have many instances without problem).

> This approach gets very slow then we are dealing with sets that have a lot features (> 100000 features)

> In scikit-learn is represented by the code: LinearRegression()

2. __Gradient Descent Method__: 

> It is an iterative optimization algorithm that updates the coefficients by moving in the direction of steepest descent as defined by the negative of the gradient. 

> The Gradient Descent Method can be faster and more scalable for large datasets, but it requires tuning the learning rate and may get stuck in local minima.

> In scikit-learn is represented by the code: SGDRegressor()


In [ ]:
#Linear Regression  (using "Normal Equation")
linear_reg = LinearRegression()

#Getting the "parameters" (also called "coefficients" or "weights") of the model using the training data. 

    #In other words, we are going to fit a linear model with "parameters" to minimize the RMSE (Root Mean Square Error) between the target value (Y-value) from the training set, and the Prediction of "Y-value".

    #In this step we are going to use the features (X-values) and target value (Y-value) from the training set.
    
linear_reg.fit(training_data_newX,training_data_targetY)

#Getting the Linear Model (in this case named as "Training_Model")

Training_Model = linear_reg.fit(training_data_newX,training_data_targetY)

In [ ]:
#Just as small test (using only the training set), we choose the first 10 elements of the training set (X values).

dataX = training_data_newX.iloc[:10]

#Then we assess if the prediction of "Y-value" [using the first 10 elements of the training set (X values)] is the same or similar to the actual target (Y-values)" (from training set)

dataY = training_data_targetY.iloc[:10]

#Prediction of "Y-value" [using the first 10 elements of the training set (X values)]

SmallPrediction_Training_Array = linear_reg.predict(dataX)
SmallPrediction_Training_Array

    #The output is very good! (those values are similar to the "Actual Y-values" (from training set)").

In [ ]:
#Transforming the previous Numpy array (called "Prediction_Training_Array") into a dataframe 

SmallPrediction_Training_DF = pd.DataFrame(SmallPrediction_Training_Array, index = [0,1,2,3,4,6,7,8,9,10])
SmallPrediction_Training_DF

In [ ]:
#Actual Y-values" (from training set)

dataY

In [ ]:
#Comparing "Actual Y values" vs "Predicted Y values" for the Training Set

    #Before the contrast, I concatenate both dataframes.

SmallComparison_Training_DF = pd.concat([SmallPrediction_Training_DF, dataY], axis=1)
SmallComparison_Training_DF.columns =['Predicted Values_Training Set', 'Actual Values_Training Set']
SmallComparison_Training_DF

## __6.4. Complete evaluation on the Training Dataset__

In [ ]:
#Prediction of "Y-value" [using all elements of the training set (X values)]

PredictionY = linear_reg.predict(training_data_newX)
PredictionY

### __6.4.1 Mean Square Error (MSE) and Root Mean Square Error (RMSE)__

__Cost Function__: it is measure of how well a ML model fits the given data. 

> It is also called "Objective Function".

> It quantifies the differences between the predicted values (Predicted Y-values) and the actual values (actual Y-values) in the data. 

> It needs to be minimized or maximized by an optimization algorithm. 

__Mean Squared Error (MSE)__: it is a specific type of "cost function" that calculates the average of the squared differences between the predicted values and the actual values. It is often used for regression problems, where the output is a continuous value.

__Root Mean Squared Error (RMSE)__: it is simply the square root of the MSE. It has the same unit as the output variable (Y-value).

> Both MSE and RMSE are more sensitive to outliers than "other cost functions", such as Mean Absolute Error (MAE).

> Both MSE and RMSE tend to favor models that have fewer parameters, which may prevent overfitting.

Just as a "__very rough guideline__" a "__good model__" has a __RMSE that is less than 10% of the range of the target variable__.

In [ ]:
#Mean Square Error (i.e. a measure of how far our predictions {from our model} is from the observed values)

linear_MSE = mean_squared_error(training_data_targetY,PredictionY)
linear_MSE

In [ ]:
#Root Mean Square Error (to deal with negative values)

linear_RMSE = np.sqrt(linear_MSE)
linear_RMSE

In [ ]:
#Transforming the previous Numpy array (called "PredictionY") into a dataframe 

Prediction_Training_DF = pd.DataFrame(PredictionY, index = training_data_newX.index)
Prediction_Training_DF

In [ ]:
#Actual Y-values" (from training set)

training_data_targetY

In [ ]:
#Comparing "Actual Y values" vs "Predicted Y values" for the Training Set

    #Before the contrast, I concatenate both dataframes.

Comparison_Training_DF = pd.concat([Prediction_Training_DF, training_data_targetY], axis=1)
Comparison_Training_DF.columns =['Predicted Values_Training Set', 'Actual Values_Training Set']
Comparison_Training_DF

In [ ]:
#Plotting just three values ("Actual Y values" vs "Predicted Y values" for the Training Set)

plt.plot(training_data_newX.index[0:3],training_data_targetY[0:3],"b.")
plt.plot(training_data_newX.index[0:3],PredictionY[0:3],"r.")
plt.xlabel("X-Axis")
plt.ylabel("Survived")
plt.title("Evaluation of the Training Set - The First three points")
plt.legend(["Training Data_Actual Y", "Training Data_Prediction Y"], loc ="lower right")
plt.show()

In [ ]:
#Plotting just all values ("Actual Y values" vs "Predicted Y values" for the Training Set)

plt.plot(training_data_newX.index,training_data_targetY,"b.")
plt.plot(training_data_newX.index,PredictionY,"r.")
plt.xlabel("X-Axis")
plt.ylabel("Survived")
plt.title("Evaluation of the Training Set - All points")
plt.legend(["Training Data_Actual Y", "Training Data_Prediction Y"], loc ="lower center")
plt.show()

# __7. Cross Validation (K-Fold Cross Validation)__

It is a statistical method that allow us:

1. Train the model in smaller "training subsets" and evaluate the model in a smaller "validation subset". In other words, the entire training set is split in smaller subsets. The aim of this method is assessing the model's behavior on unseen data. 

2. Steps:

    2.1. Split the trainig dataset into "k groups" (also called "subsets").
    
    2.2. For each unique "k group":
    
    2.2.1. Take the group as "test data set". 
    
    2.2.2. Take the remaining groups as a "training data set".
    
    2.2.3. Fit a model on the "training set" and evaluate it on the "test set"
    
    2.2.4. Save the "evaluation score" (Mean Square Error)
    
    2.3. Repeat the Steps 2.2.1 to 2.2.4
    
    2.4. Average the "evaluation scores" (Mean Square Error) of all "k groups"- 

3. To put it simply: We are going to get several  Mean Square Errors (MSE) from the training set and then we are going to averaged them, instead of getting directly only one Mean Square Error (MSE) value (as we did in the section __6.4. Complete evaluation on the Training Dataset__). In other words, it is an alternative way to do the evaluation of the model on the training set.
    
4. We can decide how many splits (i.e., "k-Folds" = "k-subsets") we want to do in the entire training set.

    4.1. The model is "trained" using "k-1" of the folds as training data. Then the model is "validated" on the remaining part of the data (i.e., the remaining "k-fold").

5. The "cross validation's performance measure" is the average of the values computed in every "validation" step.
    
    5.1. This "cross validation" is repeated multiple times, each time using a different validation subset.

6. Cross-validation methods avoid overfitting (i.e. the model learns too well from the training data and fails to generalize to new data)

7. It is useful for comparing and selecting different models or hyperparameters for a given model.

8. There are different types of cross-validation methods, such as k-fold, leave-one-out, stratified, and repeated cross-validation.

In [ ]:
#Machine Learning - Evaluation of the Model (using only the training set) by Cross Validation (K-Fold Cross Validation)

#Getting the "score" by cross-validation. <-- Very Important!

#Some relevant arguments of the "cross_val_score" function are the followings:
    #cv:  (int). cross-validation generator or an iterable, default=None, to use the default 5-fold cross validation.
    
    #scoring: (str or callable), default=None
        #It is the model-evaluation tool when we use cross-validation.
        
        #Controls what "metric" it applies to the "estimator" evaluated.
            #metric = measure the distance between the model and the data. 
                #It could be any of the predefined functions for "classification, clustering or regression" models.
                #It follows the convention that higher return values are better than lower return values (i.e., opposite to the concept of the "Cost Function").
                #In this case we are interested to find the metric "mean_squared_error" (represented by the function "neg_mean_squared_error" which returns the "negated value of the metric").
                    #Therefore, I use the minus symbol ahead "np.sqrt(-scores_training)"
            #estimators = In this case I fit the data with "linear regression" (to be specific it is called "linear_reg")

#I choose "cv = 7" considering that my total training dataset (after some transformations) has 707 values. Consequently, every fold will have around 100 values using cv=7.

scores_training = cross_val_score(linear_reg,training_data_newX,training_data_targetY, scoring="neg_mean_squared_error",cv=7)
scores_training
linear_rmse_scores = np.sqrt(-scores_training)
linear_rmse_scores

> Detailed information about how the argument "scoring" works inside the function "cross_val_score()"

<a href="https://scikit-learn.org/stable/modules/model_evaluation.html#scoring-parameter">Link about "Metrics and scoring: quantifying the quality of predictions"</a>


In [ ]:
#Showing the "mean value" of the Root Mean Square's Score by Cross Validation (K-Fold Cross Validation)
linear_rmse_scores.mean()

The previous RSME output was similar to the RSME output that we got using the training data (previous the "Cross-Validation" process) in the section __6.4.1 Mean Square Error (MSE) and Root Mean Square Error (RMSE)__

(RMSE = 0.3795654713512228) <- RMSE (previous "Cross - Validation" process)

(RMSE = 0.38485693793583964) <- RMSE (using "Cross - Validation" process)

In [ ]:
#Showing the "standard deviation" value of the Root Mean Square's Score by Cross Validation (K-Fold Cross Validation)
linear_rmse_scores.std()

# __8. Evaluation of the model on the Test Set__

After completing the following listed steps, we are going to evaluate the model on the test set.

1. Loading the data
2. Exploring the data
3. Creating a Test Set ("optional" in this problem considering that the problem has its own Test Set)
4. Visualizing the data
5. Preparing the data
6. Training the model
7. Using Cross-Validation


In [ ]:
#Final Training Model  (taken from section "6.3. Training and doing a small evaluation (on the Training Dataset)")
FinalTrainingModel = Training_Model

#Final Test Set (taken from section "6.2.5. Test Dataset - Feature Scaling (Normalization)")
FinalTestSet = test_data_norm2

#Printing the data type of the Final Training Model 
print(type(FinalTrainingModel))

#Printing the data type of the Final Test Set
print(type(FinalTestSet))

#Predictions using the "Final Test Set" in the "Final Training Model"  
FinalPrediction = FinalTrainingModel.predict(FinalTestSet)

#Printing the data type of the Predictions
print(type(FinalPrediction))

In [ ]:
#Printing the Predictions

FinalPrediction

In [ ]:
#Converting the "floats to integers" (Rounded to Nearest Integer) of Printing the Predictions.

    #The following code shows how to convert a NumPy array of floats to an array of integers in which each float is rounded to the nearest integer.

FinalPrediction2 = FinalPrediction.copy()

FinalPrediction_rounded_array = (np.rint(FinalPrediction2)).astype(int)

FinalPrediction_rounded_array

In [ ]:
#Converting the "NumPy array of integers" in a "Dataframe"

SubmissionFile1 = pd.DataFrame(FinalPrediction_rounded_array, columns = ['Survived'])

print(SubmissionFile1)
print(type(SubmissionFile1))

In [ ]:
#Joining the previous newly created dataframe (that only has the column of PassengeID) and the dataframe of the complete "Test Set"

    #Both dataframes have the same quantity of rows (418)
    
    #After the joining process, I am going to drop all the unnecessary columns (i.e. we are going to follow the rules about the submission file format requested by Kaggle)
    
        #Submission file format (according to Kaggle):  
            #The file should have exactly 2 columns:
                #PassengerId (sorted in any order)
                #Survived (contains your binary predictions: 1 for survived, 0 for deceased)

SubmissionFile2 = test_data3.join(SubmissionFile1,how="inner")

SubmissionFile2.drop(SubmissionFile2.iloc[:, 1:14], inplace=True, axis=1)
SubmissionFile2

In [ ]:
#Tranforming the file from "Pandas dataframe" to a "CSV File"

#Submission file format (according to Kaggle):  
    #Submit a csv file with exactly 418 entries plus a header row. 
    #Your submission will show an error if you have extra columns (beyond PassengerId and Survived) or rows.

Submission_FinalFile = SubmissionFile2.to_csv('Submission_FinalFile.csv', index = False)
print(Submission_FinalFile)

# __For The Record__

__I have recognized that Linear Regression is not the best way to address the problem of "Titanic" considering the following:__

1. This problem of "Titanic" has a binary target variable (i.e., taking values of 0 or 1); therefore, this problem is better addressed through a "binary classification". Two very usedul models that can be used to solve binary classification problems, are:

> Logistic Regression: model for binary classification that models the probability of the target variable being 1 as a function of the predictor variables.

> Support Vector Machines (SVMs): model that finds the hyperplane that maximally separates the two classes in the feature space.

2. The two aforementioned models are designed to model the probability of the target variable ("Y-value") taking on a particular value, which is essential in binary classification.

3. Linear regression is not a good model to deal with binary target values because:

>  It assumes that the target variable ("Y-value") is "continuous" and can take any value within a range. However, in a binary classification problem, the target variable can only take on two values (e.g., 0 or 1), which makes it a categorical variable.

> It does not model the probability of the target variable ("Y-value") taking on a particular value, which is crucial in binary classification.